# 06 — Industrialisation : Qdrant, reranker, OCR et LangGraph

**Objectif :** couvrir les étapes 1 à 3 de la feuille de route :

1. stockage vectoriel Qdrant ;
2. reranker multilingue ;
3. OCR optionnel ;
4. orchestration LangGraph.

Le serveur MCP est traité dans le notebook 05.


## Étape 1 — Qdrant local

Qdrant est utilisé en **mode local persistant** : les collections sont stockées
dans `data/processed/index/qdrant`. Les collections texte et image sont séparées.

On charge le bundle et on interroge les deux collections.


In [1]:
from maintidoc.vector_store import load_bundle

bundle = load_bundle()

for hit in bundle.text.search("Que signifie AL-01 ?", top_k=3):
    print("texte", hit["page"], hit["element_id"], round(hit["score"], 3))

for hit in bundle.image.search("orange warning LED front panel", top_k=3):
    print("image", hit["figure_id"], hit["page"], round(hit["score"], 3))

bundle.close()

texte 5 P05-TAB01 0.53
texte 7 P07-TAB03 0.488
texte 7 P07-TAB02 0.471


image FIG-02 3 0.224
image FIG-04 5 0.201
image FIG-05 6 0.163


### Lecture du résultat

On doit retrouver la page 5 dans les résultats texte et `FIG-02`/`FIG-04` dans
les résultats image. Les collections sont persistantes : elles ne sont pas
recalculées à chaque requête.


## Étape 2 — Reranker multilingue

Le reranker est un **cross-encoder** : il lit la paire (question, passage) et
produit un score de pertinence. Il est plus précis qu'un bi-encoder, mais plus
lent.

On compare l'ordre sans puis avec reranker.


In [2]:
from maintidoc.retrieval import Pipeline

pipeline = Pipeline.load_or_build(backend="qdrant")

without = pipeline.search("Que signifie le voyant orange AL-01 ?", "orange warning LED front panel", rerank=False)
with_rr = pipeline.search("Que signifie le voyant orange AL-01 ?", "orange warning LED front panel", rerank=True)

print("Sans reranker :", [h["element_id"] for h in without["text_hits"][:3]])
print("Avec reranker :", [(h["element_id"], round(h["rerank_score"], 2)) for h in with_rr["text_hits"][:3]])

pipeline.close()

Sans reranker : ['P05-TEXT', 'P05-TAB01', 'P07-TAB02']
Avec reranker : [('P05-TEXT', 4.59), ('P07-TEXT', 2.35), ('P05-TAB01', 1.47)]


### Lecture du résultat

Le reranker réordonne les passages, mais sur ce petit corpus il **dégrade** le
rappel global. Il est donc inclus mais désactivé par défaut. C'est un résultat
expérimental à documenter.


## Étape 3 — OCR optionnel

Le manuel possède une couche texte : l'OCR n'est pas nécessaire pour le texte
principal. Il est fourni pour les PDF scannés et les libellés des figures.

On teste l'OCR sur `FIG-04`.


In [3]:
from maintidoc.config import IMAGE_DIR
from maintidoc.ocr import ocr_image

texte_ocr = ocr_image(IMAGE_DIR / "fig04_page05.png")
print(texte_ocr[:800])

FlG-04|Voyants destatut(didactiques) Panneaudesupervision Representationfictive,codesnonuniversels Vert Orange Rouge Service nominal Avertissement Defaut critique Toujours verifier le code alarme, le modele et la documentation constructeur.


### Lecture du résultat

L'OCR détecte les libellés `Vert`, `Orange`, `Rouge`, `Service nominal`,
`Avertissement`, `Défaut critique`. Le texte est utile mais imparfait :
certains espaces manquent. Il ne remplace pas une couche texte native.


## Étape 4 — LangGraph

Le graphe enchaîne :

```text
retrieve → generate → validate
                         ├─ answer
                         └─ clarify
```

On lance l'agent en anglais, comme dans l'application.


In [4]:
from maintidoc.graph import run_agent

result = run_agent(
    "What does the orange AL-01 LED mean?",
    "orange warning LED storage array front panel",
    model="llama3.2:latest",
    language="en",
)
print("Statut :", result["answer"]["status"])
print(result["answer"]["answer"])
print(result["answer"].get("citations"))

Statut : answered
The orange AL-01 LED indicates a degraded FC (Fibre Channel) path on HBA1. This requires verification of DIAG-FC-01 and FIG-02.
[{'id': 'P05-TAB01', 'page': 5}, {'id': 'P05-TEXT', 'page': 5}, {'id': 'P07-TAB03', 'page': 7}, {'id': 'P07-TAB02', 'page': 7}, {'id': 'P07-TEXT', 'page': 7}]


### Lecture du résultat

Le statut doit être `answered` avec une citation page 5. Le graphe a exécuté les
nœuds `retrieve`, `generate` et `validate`.


## Étape 5 — Clarification

Une question sans réponse doit être routée vers `clarify`, pas inventée.


In [5]:
clarify = run_agent(
    "What is the price of the SAN-LAB 2000?",
    "storage array price",
    model="llama3.2:latest",
    language="en",
)
print("Statut :", clarify["answer"]["status"])
print(clarify["answer"]["answer"])

Statut : needs_clarification
I cannot support a reliable answer with the retrieved evidence. Please provide the exact alarm code, the equipment model, or a sharper photo of the panel.


### Lecture du résultat

Le statut doit être `needs_clarification` ou `insufficient_evidence`, avec une
demande de précision. C'est le comportement attendu face à une preuve absente.


## Bilan

| Brique | État |
|---|---|
| Qdrant texte + image | fait |
| reranker multilingue | fait, optionnel |
| OCR figures | fait, optionnel |
| LangGraph | fait |
| MCP | notebook 05 |

**Étape suivante :** FastAPI, authentification/RBAC, 150 scénarios, fine-tuning.
